In [ ]:
# ⚠️ Takes 1-2 mins to download
from sentence_transformers import CrossEncoder
import os

RERANKER_MODEL = os.environ.get("RERANKER_MODEL", "cross-encoder/ms-marco-MiniLM-L-6-v2")
print(f"⏳ Loading reranker: {RERANKER_MODEL}")
reranker = CrossEncoder(RERANKER_MODEL)
print("✅ Reranker loaded")

def rerank_results(query: str, results: list, top_k: int = 5) -> list:
    """Cross-encoder reranking."""
    if not results or len(results) == 1:
        return results

    pairs = [(query, r.content) for r in results]
    scores = reranker.predict(pairs)

    for r, score in zip(results, scores):
        r.score = float(score)

    reranked = sorted(results, key=lambda x: x.score, reverse=True)
    return reranked[:top_k]

# Test reranker
if 'all_results' in dir() and all_results:
    reranked = rerank_results(query, all_results[:20], top_k=5)
    print("\n🔄 After Reranking (top 5):")
    for i, r in enumerate(reranked):
        print(f"  {i+1}. Score: {r.score:.4f} | {r.content[:100]}...")

⏳ Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Reranker loaded


In [13]:
# To use the OpenAI API, you'll need an API key.
# Add it to the Colab secrets manager under the "🔑" in the left panel. Give it the name `OPENAI_API_KEY`.

from google.colab import userdata
import openai

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

client = openai.OpenAI(api_key=OPENAI_API_KEY)
MODEL = 'gpt-3.5-turbo'

print("✅ OpenAI client and model initialized.")

SecretNotFoundError: Secret OPENAI_API_KEY does not exist.

In [1]:
import re

def format_context(results: list) -> str:
    parts = []
    for i, r in enumerate(results):
        source = r.metadata.get("source", "Unknown")
        parts.append(f"[Doc {i+1}] (Source: {source})\n{r.content}")
    return "\n\n---\n\n".join(parts)

def extract_citations(answer: str, results: list) -> list:
    citations = []
    for match in re.finditer(r'\[Doc (\d+)\]', answer):
        idx = int(match.group(1)) - 1
        if 0 <= idx < len(results):
            citations.append({
                "doc_num": idx + 1,
                "chunk_id": results[idx].chunk_id,
                "source": results[idx].metadata.get("source", ""),
                "snippet": results[idx].content[:200]
            })
    return citations

def generate_answer(query: str, context_results: list, history: list = None) -> dict:
    """Generate answer with citations."""
    context = format_context(context_results)

    messages = [{
        "role": "system",
        "content": """You are a precise, factual assistant.
Answer ONLY based on the provided context.
Use [Doc N] citations for every claim.
If context is insufficient, say so clearly."""
    }]

    if history:
        messages.extend(history[-4:])

    messages.append({"role": "user", "content": f"""
Context:
{context}

Question: {query}

Instructions:
- Answer using ONLY the context above
- Cite every claim as [Doc N]
- Be concise but complete
- If info is missing, acknowledge it"""})

    response = client.chat.completions.create(
        model=MODEL, messages=messages,
        temperature=0.2, max_tokens=1000
    )

    answer = response.choices[0].message.content
    return {
        "query": query,
        "answer": answer,
        "citations": extract_citations(answer, context_results),
        "tokens_used": response.usage.total_tokens,
        "context_chunks": [r.content for r in context_results]
    }

print("✅ Generator ready")

✅ Generator ready


In [3]:
import collections

# Placeholder for a search result
SearchResult = collections.namedtuple('SearchResult', ['chunk_id', 'content', 'metadata', 'score'])

def smart_rewrite(query: str):
    """Placeholder for smart query rewriting."""
    # This should be replaced with actual query rewriting logic
    return type('RewriteResult', (object,), {
        'strategy_used': 'no_rewrite',
        'all_queries': [query]
    })()

def rewrite_multi_query(query: str):
    """Placeholder for multi-query rewriting."""
    # This should be replaced with actual multi-query rewriting logic
    return type('RewriteResult', (object,), {
        'strategy_used': 'multi_query',
        'all_queries': [query, query + ' details'] # Example for multiple queries
    })()

def hybrid_search(query: str, top_k: int) -> list:
    """Placeholder for hybrid search, returning dummy results."""
    # Replace with actual retrieval logic (e.g., vector DB, keyword search)
    dummy_results = [
        SearchResult(f'chunk_{i}', f'This is a dummy content chunk related to {query}. Result {i}.', {'source': f'doc_{i}.pdf'}, 0.5 + i*0.01)
        for i in range(top_k)
    ]
    return dummy_results

def evaluate_retrieval(query: str, results: list):
    """Placeholder for retrieval evaluation."""
    # Replace with actual evaluation logic
    return type('EvalResult', (object,), {
        'overall_score': 0.8,
        'suggested_action': 'none',
        'filtered_results': results # For now, no filtering
    })()

print("✅ Placeholder functions for rewriting, search, and evaluation are defined.")

✅ Placeholder functions for rewriting, search, and evaluation are defined.


In [8]:
# ⚠️ Takes 1-2 mins to download
from sentence_transformers import CrossEncoder
import os

RERANKER_MODEL = os.environ.get("RERANKER_MODEL", "cross-encoder/ms-marco-MiniLM-L-6-v2")
print(f"⏳ Loading reranker: {RERANKER_MODEL}")
reranker = CrossEncoder(RERANKER_MODEL)
print("✅ Reranker loaded")

def rerank_results(query: str, results: list, top_k: int = 5) -> list:
    """Cross-encoder reranking."""
    if not results or len(results) == 1:
        return results

    pairs = [(query, r.content) for r in results]
    scores = reranker.predict(pairs)

    # Namedtuples are immutable, so we create new SearchResult instances with updated scores
    updated_results = []
    for r, score in zip(results, scores):
        updated_results.append(r._replace(score=float(score)))

    reranked = sorted(updated_results, key=lambda x: x.score, reverse=True)
    return reranked[:top_k]

# Test reranker (optional, only runs if 'all_results' is defined)
if 'all_results' in dir() and all_results:
    reranked = rerank_results(query, all_results[:20], top_k=5)
    print("\n🔄 After Reranking (top 5):")
    for i, r in enumerate(reranked):
        print(f"  {i+1}. Score: {r.score:.4f} | {r.content[:100]}...")

⏳ Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Reranker loaded


In [16]:
def full_rag_pipeline(query: str, verbose: bool = True) -> dict:
    """Complete RAG pipeline: rewrite → retrieve → evaluate → rerank → generate."""

    if verbose: print(f"\n{'='*60}\nQuery: {query}\n{'='*60}")

    # 1. Rewrite
    rewritten = smart_rewrite(query)
    if verbose: print(f"✅ Step 1 - Strategy: {rewritten.strategy_used}")

    # 2. Retrieve
    all_results = []
    seen = set()
    for q in rewritten.all_queries[:3]:
        for r in hybrid_search(q, top_k=15):
            if r.chunk_id not in seen:
                seen.add(r.chunk_id)
                all_results.append(r)
    if verbose: print(f"✅ Step 2 - Retrieved: {len(all_results)} chunks")

    # 3. Evaluate retrieval
    eval_result = evaluate_retrieval(query, all_results)
    if verbose: print(f"✅ Step 3 - Quality: {eval_result.overall_score:.2f} ({eval_result.suggested_action})")

    # 3b. Retry if poor (simple loop, no LangGraph yet)
    retry_count = 0
    if eval_result.suggested_action == "requery" and retry_count < 2:
        retry_rewritten = rewrite_multi_query(query)
        retry_results = []
        for q in retry_rewritten.all_queries[:3]:
            for r in hybrid_search(q, top_k=15):
                if r.chunk_id not in seen:
                    seen.add(r.chunk_id)
                    retry_results.append(r)
        all_results.extend(retry_results)
        retry_count += 1
        if verbose: print(f"   ↩️ Retried retrieval, now have {len(all_results)} chunks")

    # 4. Rerank
    candidates = eval_result.filtered_results or all_results
    reranked = rerank_results(query, candidates[:20], top_k=5)
    if verbose: print(f"✅ Step 4 - Reranked to top {len(reranked)}")

    # 5. Generate
    answer_data = generate_answer(query, reranked)
    if verbose:
        print(f"✅ Step 5 - Generated ({answer_data['tokens_used']} tokens)")
        print(f"\n📝 ANSWER:\n{answer_data['answer']}")
        print(f"\n📚 Citations: {len(answer_data['citations'])}")

    return {**answer_data, "rewrite_strategy": rewritten.strategy_used,
            "retrieval_score": eval_result.overall_score, "retry_count": retry_count}

# Run it!
result = full_rag_pipeline("What is retrieval augmented generation and how does it work?")


Query: What is retrieval augmented generation and how does it work?
✅ Step 1 - Strategy: no_rewrite
✅ Step 2 - Retrieved: 15 chunks
✅ Step 3 - Quality: 0.80 (none)
✅ Step 4 - Reranked to top 5


NameError: name 'client' is not defined

In [26]:
# ============================================================
# CELL 0 — Run this FIRST before any other cell in Notebook 7
# ============================================================
import os, json, re
from google.colab import userdata
from tenacity import retry, stop_after_attempt, wait_exponential
from dataclasses import dataclass, field
from typing import List, Optional

# Install groq if not already installed
try:
    import groq
except ImportError:
    print("Installing groq library...")
    !pip install groq
    import groq

# Install faiss-cpu if not already installed
try:
    import faiss
except ImportError:
    print("Installing faiss-cpu library...")
    !pip install faiss-cpu
    import faiss

# Install rank_bm25 if not already installed
try:
    import rank_bm25
except ImportError:
    print("Installing rank_bm25 library...")
    !pip install rank_bm25
    import rank_bm25

# ── 1. Load API key and create client ────────────────────────
try:
    api_key = userdata.get("GROQ_API_KEY")
    if api_key:
        os.environ["GROQ_API_KEY"] = api_key
        from groq import Groq
        client = Groq(api_key=api_key)
        MODEL  = "llama-3.3-70b-versatile"
        print("✅ Groq client ready")
    else:
        raise ValueError("Empty Groq key")
except Exception as groq_err:
    # Fallback to OpenAI if Groq key not set
    try:
        api_key = userdata.get("OPENAI_API_KEY")
        os.environ["OPENAI_API_KEY"] = api_key
        from openai import OpenAI
        client = OpenAI(api_key=api_key)
        MODEL  = "gpt-4o-mini"
        print("✅ OpenAI client ready")
    except Exception as oai_err:
        raise EnvironmentError(f"No working API key found.\nGroq error: {groq_err}\nOpenAI error: {oai_err}")

# Quick test
test = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say READY in one word"}],
    max_tokens=5
)
print(f"✅ API test passed → {test.choices[0].message.content.strip()}")

# ── 2. Reload all search indexes from Drive ───────────────────
import numpy as np, pickle, json as json_lib
from pathlib import Path

PROCESSED_DIR = "/content/drive/MyDrive/SelfImprovingRAG/data/processed"

# Define Chunk dataclass (missing definition was causing AttributeError during pickle load)
@dataclass
class Chunk:
    chunk_id: str
    content:  str
    metadata: dict = field(default_factory=dict)
    parent_chunk_id: Optional[str] = None

# Load chunks
with open(f"{PROCESSED_DIR}/parent_chunks.pkl", "rb") as f:
    all_parent_chunks = pickle.load(f)
with open(f"{PROCESSED_DIR}/child_chunks.pkl",  "rb") as f:
    all_child_chunks  = pickle.load(f)

# Load embeddings + IDs
all_embeddings = np.load(f"{PROCESSED_DIR}/child_embeddings.npy")
with open(f"{PROCESSED_DIR}/child_ids.json") as f:
    child_ids = json_lib.load(f)

# Load BM25
with open(f"{PROCESSED_DIR}/bm25_index.pkl", "rb") as f:
    bm25_data = pickle.load(f)
bm25     = bm25_data["bm25"]
bm25_ids = bm25_data["chunk_ids"]

# Rebuild FAISS in memory (always fast, ~1 sec)
dim         = all_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
emb_copy    = all_embeddings.copy().astype(np.float32)
faiss.normalize_L2(emb_copy)
faiss_index.add(emb_copy)
chunk_map = {c.chunk_id: c for c in all_child_chunks}

print(f"✅ FAISS : {faiss_index.ntotal} vectors | dim={dim}")
print(f"✅ BM25  : {len(bm25_ids)} docs")
print(f"✅ Chunks: {len(chunk_map)} child | {len(all_parent_chunks)} parent")

# ── 3. Reload the embedding model ────────────────────────────
from sentence_transformers import SentenceTransformer
EMBED_MODEL_NAME = os.environ.get("EMBEDDING_MODEL", "BAAI/bge-base-en-v1.5")
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print(f"✅ Embedding model loaded: {EMBED_MODEL_NAME}")

# ── 4. Redefine search helper functions ──────────────────────
STOPWORDS = {"the","a","an","and","or","but","in","on","at","to","for",
             "of","with","is","was","are","were","be","been","have","has",
             "had","do","does","did","will","would","could","should","may",
             "might","this","that","it","its"}

def tokenize(text):
    text = re.sub(r'[^\w\s]', ' ', text.lower())
    return [t for t in text.split() if t not in STOPWORDS and len(t) > 1]

@dataclass
class SearchResult:
    chunk_id: str
    content:  str
    score:    float
    metadata: dict   = field(default_factory=dict)
    parent_chunk_id:  Optional[str] = None
    bm25_rank:  int  = 0
    vector_rank:int  = 0

def hybrid_search(query: str, top_k: int = 10) -> List[SearchResult]:
    K = 60
    tokens      = tokenize(query)
    bm25_scores = bm25.get_scores(tokens)
    bm25_top    = bm25_scores.argsort()[::-1][:top_k * 3]
    bm25_ranked = [(bm25_ids[i], float(bm25_scores[i])) for i in bm25_top if bm25_scores[i] > 0]

    prefix  = "Represent this sentence for searching relevant passages: "
    q_emb   = embed_model.encode([prefix + query],
                                  normalize_embeddings=True,
                                  convert_to_numpy=True).astype(np.float32)
    D, I    = faiss_index.search(q_emb, k=top_k * 3)
    vec_ranked = [(bm25_ids[idx], float(D[0][r]))
                  for r, idx in enumerate(I[0]) if idx >= 0]

    rrf = {}
    for rank, (cid, _) in enumerate(bm25_ranked):
        rrf[cid] = rrf.get(cid, 0) + 0.3 / (K + rank + 1)
    for rank, (cid, _) in enumerate(vec_ranked):
        rrf[cid] = rrf.get(cid, 0) + 0.7 / (K + rank + 1)

    sorted_ids = sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:top_k]
    results    = []
    for rank, (cid, rrf_score) in enumerate(sorted_ids):
        if cid in chunk_map:
            c = chunk_map[cid]
            br = next((i for i,(id_,_) in enumerate(bm25_ranked) if id_==cid), 999)
            vr = next((i for i,(id_,_) in enumerate(vec_ranked)  if id_==cid), 999)
            results.append(SearchResult(
                chunk_id=cid, content=c.content, score=rrf_score,
                metadata=c.metadata, parent_chunk_id=c.parent_chunk_id,
                bm25_rank=br, vector_rank=vr
            ))
    return results

print("\n✅ Everything loaded — ready to run all cells in Notebook 7")
print("▶  Continue with reranker cell next")

Installing rank_bm25 library...
✅ Groq client ready
✅ API test passed → READY
✅ FAISS : 5 vectors | dim=768
✅ BM25  : 5 docs
✅ Chunks: 5 child | 1 parent


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded: BAAI/bge-base-en-v1.5

✅ Everything loaded — ready to run all cells in Notebook 7
▶  Continue with reranker cell next


In [23]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
